# Prepare
Build a symbol universe from the input CSV, load each `{symbol}.csv` archive, fetch missing EOD rows from EODHD, merge/deduplicate, and save updated per-symbol CSV files.

## Imports

### Purpose
Centralize all dependencies used throughout the notebook so later cells can focus on business logic only.

### Imported modules

| Module | Role in pipeline |
|---|---|
| `requests` | EODHD HTTP calls |
| `pandas` | DataFrame transforms, CSV I/O, merge/dedupe |
| `pathlib.Path` | File path resolution and archive mapping |
| `datetime` (`date`, `timedelta`) | Incremental window calculations |
| `math` | Chunk-count math for batch orchestration |
| `os` | Read/write access checks |
| `sys` | Runtime diagnostics |
| `time` | Retry back-off timing |
| `importlib.util` | Dynamic loading of `config.py` |
| `typing` | Type annotations for helper functions |

### Output
All required modules are available in the global notebook runtime.

In [1]:
import importlib.util
import math
import os
import sys
import time
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import requests

## Block 2 — Configuration

### Purpose
Define all runtime settings in one place and fail early on invalid setup.

### Runtime parameters

| Parameter | Meaning |
|---|---|
| `CSV_PATH` | Input ticker CSV path (currently `data/DEMO_003.csv` for development) |
| `CONFIG_EODHD_API_KEY` | Key read from `config.py` for validation |
| `EODHD_API_KEY` | Key used for HTTP calls (currently forced to `'demo'`) |
| `REQUEST_TIMEOUT` | HTTP timeout in seconds |
| `MAX_RETRIES` | Retry attempts for transient failures |
| `DRY_RUN` | If `True`, skips file writes |
| `BATCH_SIZE` | Incremental chunk size (fixed to 100) |

### Validation performed
- Input CSV exists and is readable.
- `config.py` contains a non-empty `EODHD_API_KEY`.
- Scope folder is creatable/writable.
- Safe configuration summary is printed without exposing secret values.

### Output
Validated runtime config variables used by all downstream blocks.

In [2]:
# Runtime location discovery
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "prepare.ipynb").exists():
    # Assumption: kernel may be started from a parent directory.
    candidates = list(NOTEBOOK_DIR.glob("**/prepare/prepare.ipynb"))
    if candidates:
        NOTEBOOK_DIR = candidates[0].parent.resolve()

REPO_ROOT = NOTEBOOK_DIR.parent

# Development-mode defaults requested by user
CSV_PATH = (REPO_ROOT / "data" / "DEMO_003.csv").resolve()
REQUEST_TIMEOUT = 30
MAX_RETRIES = 3
DRY_RUN = False
BATCH_SIZE = 100

# Read API key from config.py as required, but force demo key for development calls.
CONFIG_PATH = (REPO_ROOT.parent / "starter_files" / "config.py").resolve()
CONFIG_EODHD_API_KEY = ""
if CONFIG_PATH.exists():
    spec = importlib.util.spec_from_file_location("project_config", str(CONFIG_PATH))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    CONFIG_EODHD_API_KEY = str(getattr(module, "EODHD_API_KEY", "")).strip()

EODHD_API_KEY = "demo"

# Validation
if not CSV_PATH.exists() or not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV_PATH does not exist or is not a file: {CSV_PATH}")
if not os.access(CSV_PATH, os.R_OK):
    raise PermissionError(f"CSV_PATH is not readable: {CSV_PATH}")
if not CONFIG_EODHD_API_KEY:
    raise ValueError(f"Missing or empty EODHD_API_KEY in config file: {CONFIG_PATH}")

LIST_NAME_PREVIEW = CSV_PATH.stem
SCOPE_FOLDER_PREVIEW = CSV_PATH.parent / LIST_NAME_PREVIEW
SCOPE_FOLDER_PREVIEW.mkdir(parents=True, exist_ok=True)
if not os.access(SCOPE_FOLDER_PREVIEW, os.W_OK):
    raise PermissionError(f"Archive folder is not writable: {SCOPE_FOLDER_PREVIEW}")

print("Configuration validated:")
print(f"  Python version: {sys.version.split()[0]}")
print(f"  CSV_PATH: {CSV_PATH}")
print(f"  REQUEST_TIMEOUT: {REQUEST_TIMEOUT}s | MAX_RETRIES: {MAX_RETRIES}")
print(f"  DRY_RUN: {DRY_RUN} | BATCH_SIZE: {BATCH_SIZE}")
print(f"  Scope folder: {SCOPE_FOLDER_PREVIEW}")
print("  API key source: config.py (read successfully)")
print("  API key in use for development: demo")

Configuration validated:
  Python version: 3.9.24
  CSV_PATH: C:\Users\fraggable\DCC_Created\Coding\algorithmic-trading-python\1‑month-direction-classifier\data\DEMO_003.csv
  REQUEST_TIMEOUT: 30s | MAX_RETRIES: 3
  DRY_RUN: False | BATCH_SIZE: 100
  Scope folder: C:\Users\fraggable\DCC_Created\Coding\algorithmic-trading-python\1‑month-direction-classifier\data\DEMO_003
  API key source: config.py (read successfully)
  API key in use for development: demo


## Block 3 — Load And Validate Ticker Universe

### Purpose
Load symbols from `CSV_PATH`, normalize them, and produce a deterministic symbol list for this run.

### Processing rules
1. Prefer a `Symbol` column (case-insensitive); otherwise use the first column.
2. Strip whitespace and convert symbols to uppercase.
3. Remove empty values.
4. Remove duplicates while preserving first occurrence order.

### Diagnostics
- Source row count.
- Final symbol count.
- Removed-row report (empty/duplicate), including row context.

### Output
- `symbols: list[str]` — canonical symbol universe.
- `symbol_diagnostics: DataFrame` — removed-row diagnostics.

In [3]:
def load_ticker_universe(csv_path: Path) -> Tuple[List[str], pd.DataFrame]:
    raw_df = pd.read_csv(csv_path, dtype=str)
    if raw_df.empty:
        return [], pd.DataFrame(columns=["reason", "value"])

    symbol_column: Optional[str] = None
    for col in raw_df.columns:
        if str(col).strip().lower() == "symbol":
            symbol_column = col
            break

    if symbol_column is None:
        symbol_column = raw_df.columns[0]

    cleaned_values: List[str] = []
    diagnostics: List[Dict[str, Any]] = []
    seen: set = set()

    for idx, raw_value in enumerate(raw_df[symbol_column].tolist()):
        normalized = str(raw_value).strip().upper() if pd.notna(raw_value) else ""

        if normalized == "" or normalized == "NAN":
            diagnostics.append({"reason": "empty", "value": raw_value, "row": idx})
            continue

        if normalized in seen:
            diagnostics.append({"reason": "duplicate", "value": normalized, "row": idx})
            continue

        seen.add(normalized)
        cleaned_values.append(normalized)

    diag_df = pd.DataFrame(diagnostics)
    return cleaned_values, diag_df

symbols, symbol_diagnostics = load_ticker_universe(CSV_PATH)

print(f"Ticker rows in source: {len(pd.read_csv(CSV_PATH, dtype=str))}")
print(f"Symbols after normalization and dedupe: {len(symbols)}")
if symbol_diagnostics.empty:
    print("No invalid or duplicate rows were removed.")
else:
    print(f"Removed rows: {len(symbol_diagnostics)}")
    print(symbol_diagnostics.to_string(index=False))

if not symbols:
    raise ValueError("No valid symbols were found in the selected CSV.")

Ticker rows in source: 3
Symbols after normalization and dedupe: 3
No invalid or duplicate rows were removed.


## Block 4 — Resolve Scope Folder And Build Archive Map

### Purpose
Derive the deterministic archive destination for this ticker scope and map each symbol to one CSV path.

### Rules
- `LIST_NAME = CSV_PATH.stem`.
- `SCOPE_FOLDER = data/LIST_NAME/`.
- `archive_map[symbol] = SCOPE_FOLDER / f"{symbol}.csv"`.
- Create `SCOPE_FOLDER` if missing.

### Output
- `LIST_NAME: str`
- `SCOPE_FOLDER: Path`
- `archive_map: dict[str, Path]`

This mapping is the canonical file registry for bootstrap, incremental fetch, merge, and reporting.

In [4]:
def derive_scope_paths(csv_path: Path, symbol_list: List[str]) -> Tuple[str, Path, Dict[str, Path]]:
    list_name = csv_path.stem
    scope_folder = (csv_path.parent / list_name).resolve()
    scope_folder.mkdir(parents=True, exist_ok=True)
    symbol_map = {symbol: scope_folder / f"{symbol}.csv" for symbol in symbol_list}
    return list_name, scope_folder, symbol_map

LIST_NAME, SCOPE_FOLDER, archive_map = derive_scope_paths(CSV_PATH, symbols)

print(f"LIST_NAME: {LIST_NAME}")
print(f"SCOPE_FOLDER: {SCOPE_FOLDER}")
print(f"Mapped symbols: {len(archive_map)}")
if archive_map:
    first_symbol = next(iter(archive_map))
    print(f"Example mapping: {first_symbol} -> {archive_map[first_symbol]}")

LIST_NAME: DEMO_003
SCOPE_FOLDER: C:\Users\fraggable\DCC_Created\Coding\algorithmic-trading-python\1‑month-direction-classifier\data\DEMO_003
Mapped symbols: 3
Example mapping: AAPL.US -> C:\Users\fraggable\DCC_Created\Coding\algorithmic-trading-python\1‑month-direction-classifier\data\DEMO_003\AAPL.US.csv


## Block 5 — Discover Existing Vs New Symbols

### Purpose
Split the symbol universe into symbols with existing archives and symbols that require first-time initialization.

### Logic
For each symbol path in `archive_map`:
- If file exists: add to `existing_symbols`.
- If file is missing: add to `new_symbols`.

### Why this matters
New symbols must be bootstrapped with full history before shared incremental processing can be correct.

### Output
- `existing_symbols: list[str]`
- `new_symbols: list[str]`
- Run-level tracking containers initialized for failures, statuses, payloads, and metrics.

In [5]:
def detect_new_symbols(symbol_list: List[str], symbol_map: Dict[str, Path]) -> Tuple[List[str], List[str]]:
    existing = [symbol for symbol in symbol_list if symbol_map[symbol].exists()]
    new = [symbol for symbol in symbol_list if not symbol_map[symbol].exists()]
    return existing, new

existing_symbols, new_symbols = detect_new_symbols(symbols, archive_map)

# Run-level tracking containers used by downstream blocks.
bootstrap_failures: Dict[str, str] = {}
bootstrap_success_symbols: List[str] = []
precheck_failures: Dict[str, str] = {}
symbol_failures: Dict[str, str] = {}
failed_chunks: List[Dict[str, Any]] = []
incremental_payload: Dict[str, List[Dict[str, Any]]] = {symbol: [] for symbol in symbols}
merge_stats: Dict[str, Dict[str, int]] = {}
status_by_symbol: Dict[str, str] = {}
skip_phase2 = False

print(f"Total symbols: {len(symbols)}")
print(f"Existing symbols: {len(existing_symbols)}")
print(f"New symbols: {len(new_symbols)}")
if new_symbols:
    print("New symbols to bootstrap:")
    print(", ".join(new_symbols))
else:
    print("No new symbols detected.")

Total symbols: 3
Existing symbols: 3
New symbols: 0
No new symbols detected.


## Block 6 — Bootstrap New Symbols (Full History)

### Purpose
Initialize missing archives before incremental updates begin.

### Processing
For each symbol in `new_symbols`:
1. Call `GET /eod/<SYMBOL>` without date bounds.
2. Normalize payload into canonical columns: `Date, Open, High, Low, Close, Adjusted_close, Volume`.
   - Use `normalize_payload_to_rows` if already available.
   - Otherwise use this block’s fallback normalizer.
3. Write archive CSV to `archive_map[symbol]` (unless `DRY_RUN=True`).
4. On error, record failure and continue.

### Reliability behavior
- Retries are applied (`MAX_RETRIES`) with exponential back-off.
- A single symbol failure never aborts the phase.

### Output
- Bootstrapped archive files for successful symbols.
- `bootstrap_success_symbols` and `bootstrap_failures`.
- Per-symbol progress logs.

In [6]:
def _normalize_records_fallback(raw_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    canonical_cols = ["Date", "Open", "High", "Low", "Close", "Adjusted_close", "Volume"]
    if not raw_rows:
        return pd.DataFrame(columns=canonical_cols)

    df = pd.DataFrame(raw_rows)
    rename_map = {
        "date": "Date",
        "Date": "Date",
        "open": "Open",
        "high": "High",
        "low": "Low",
        "close": "Close",
        "adjusted_close": "Adjusted_close",
        "adjustedClose": "Adjusted_close",
        "adjusted close": "Adjusted_close",
        "volume": "Volume",
    }
    df = df.rename(columns={col: rename_map.get(col, col) for col in df.columns})

    for col in canonical_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.strftime("%Y-%m-%d")
    for col in ["Open", "High", "Low", "Close", "Adjusted_close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Date", "Close"]).copy()
    return df[canonical_cols].sort_values("Date").reset_index(drop=True)


def fetch_full_history_for_symbol(symbol: str) -> List[Dict[str, Any]]:
    url = f"https://eodhd.com/api/eod/{symbol}"
    params = {"api_token": EODHD_API_KEY, "fmt": "json"}
    last_error: Optional[Exception] = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            payload = response.json()
            if isinstance(payload, dict) and payload.get("error"):
                raise RuntimeError(str(payload.get("error")))
            if not isinstance(payload, list):
                raise ValueError("Unexpected payload shape for full-history endpoint.")
            return payload
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(2 ** (attempt - 1))

    raise RuntimeError(f"Bootstrap request failed for {symbol}: {last_error}")


def bootstrap_new_symbols(symbol_list: List[str]) -> None:
    if not symbol_list:
        print("No new symbols to bootstrap.")
        return

    for symbol in symbol_list:
        try:
            rows = fetch_full_history_for_symbol(symbol)
            maybe_normalizer = globals().get("normalize_payload_to_rows")
            if callable(maybe_normalizer):
                normalized_df = maybe_normalizer(rows)
            else:
                # Block 9 runs later, so bootstrap uses a local normalizer for now.
                normalized_df = _normalize_records_fallback(rows)

            if normalized_df.empty:
                raise ValueError("API returned no usable rows after normalization.")

            target_path = archive_map[symbol]
            if not DRY_RUN:
                normalized_df.to_csv(target_path, index=False)

            merge_stats[symbol] = {
                "rows_before": 0,
                "rows_after": int(len(normalized_df)),
                "rows_added": int(len(normalized_df)),
            }
            bootstrap_success_symbols.append(symbol)
            print(f"BOOTSTRAP OK  | {symbol} | rows={len(normalized_df)}")
        except Exception as exc:
            reason = str(exc)
            bootstrap_failures[symbol] = reason
            symbol_failures[symbol] = reason
            print(f"BOOTSTRAP FAIL| {symbol} | {reason}")


bootstrap_new_symbols(new_symbols)
print(f"Bootstrap complete: {len(bootstrap_success_symbols)} succeeded, {len(bootstrap_failures)} failed")

No new symbols to bootstrap.
Bootstrap complete: 0 succeeded, 0 failed


## Block 7 — Compute Shared Incremental Window

### Purpose
Calculate one deterministic incremental date window for Phase 2.

### Logic
1. Read each symbol archive and extract its latest valid `Date`.
2. Build `latest_dates: dict[str, date]`.
3. Set `shared_from = min(latest_dates.values())`.
4. Set `shared_to = date.today()`.

### Edge handling
- Missing/empty/invalid archives are tracked in precheck failures and excluded from minimum-date calculation.
- If no valid dates exist, or `shared_from >= shared_to`, set `skip_phase2=True`.

### Output
- `latest_dates`, `shared_from`, `shared_to`, `skip_phase2`.
- Printed shared-window diagnostics.

In [7]:
def read_latest_dates(symbol_map: Dict[str, Path]) -> Tuple[Dict[str, date], Dict[str, str]]:
    latest: Dict[str, date] = {}
    failures: Dict[str, str] = {}

    for symbol, file_path in symbol_map.items():
        if not file_path.exists():
            failures[symbol] = "Archive file missing after bootstrap phase."
            continue

        try:
            df = pd.read_csv(file_path)
            if df.empty:
                failures[symbol] = "Archive file is empty."
                continue
            if "Date" not in df.columns:
                failures[symbol] = "Archive missing required Date column."
                continue

            parsed_dates = pd.to_datetime(df["Date"], errors="coerce")
            if parsed_dates.isna().all():
                failures[symbol] = "Archive contains no valid Date values."
                continue

            latest[symbol] = parsed_dates.max().date()
        except Exception as exc:
            failures[symbol] = f"Failed reading archive: {exc}"

    return latest, failures


latest_dates, precheck_failures = read_latest_dates(archive_map)
symbol_failures.update({k: v for k, v in precheck_failures.items() if k not in symbol_failures})

shared_from: Optional[date] = None
shared_to: date = date.today()

if latest_dates:
    shared_from = min(latest_dates.values())
    window_days = (shared_to - shared_from).days
    skip_phase2 = shared_from >= shared_to
    print(f"Shared window: from={shared_from} to={shared_to} ({window_days} calendar days)")
    if skip_phase2:
        print("Phase 2 skipped: all archives are already current for today.")
else:
    skip_phase2 = True
    print("Phase 2 skipped: no valid latest dates available.")

if precheck_failures:
    print(f"Precheck flagged {len(precheck_failures)} symbol(s):")
    for symbol, reason in precheck_failures.items():
        print(f"  - {symbol}: {reason}")

Shared window: from=2026-04-10 to=2026-04-18 (8 calendar days)


## Block 8 — Fetch Incremental Data In 100-Symbol Chunks

### Purpose
Retrieve shared-window incremental rows for all eligible symbols with retry-aware chunk orchestration.

### Orchestration
- Build chunks of size `BATCH_SIZE` (100).
- Process chunks in stable symbol order.
- Retry each chunk up to `MAX_RETRIES`, back-off exponentially.

### Current request strategy
- In demo-mode implementation, each symbol in a chunk is fetched via `GET /eod/<SYMBOL>` with `from`/`to`.
- Successful rows are accumulated in `incremental_payload[symbol]`.
- Unresolved symbol errors are recorded in `failed_chunks` and `symbol_failures`.

### Output
- `incremental_payload: dict[str, list[dict]]`
- `failed_chunks: list[dict]`
- Updated `symbol_failures`
- Chunk-level progress logs.

In [8]:
def chunk_symbols(symbol_list: List[str], chunk_size: int = 100) -> List[List[str]]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be > 0")
    return [symbol_list[i : i + chunk_size] for i in range(0, len(symbol_list), chunk_size)]


def fetch_incremental_batch(symbol_batch: List[str], from_date: date, to_date: date) -> Tuple[Dict[str, List[Dict[str, Any]]], Dict[str, str]]:
    batch_payload: Dict[str, List[Dict[str, Any]]] = {}
    batch_errors: Dict[str, str] = {}

    # Assumption: demo mode is most reliable with per-symbol /eod requests.
    # We still preserve 100-symbol chunk orchestration for deterministic retries/reporting.
    for symbol in symbol_batch:
        url = f"https://eodhd.com/api/eod/{symbol}"
        params = {
            "api_token": EODHD_API_KEY,
            "fmt": "json",
            "from": from_date.isoformat(),
            "to": to_date.isoformat(),
        }
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            payload = response.json()
            if isinstance(payload, dict) and payload.get("error"):
                raise RuntimeError(str(payload.get("error")))
            if not isinstance(payload, list):
                raise ValueError("Unexpected payload shape.")
            batch_payload[symbol] = payload
        except Exception as exc:
            batch_errors[symbol] = str(exc)

    return batch_payload, batch_errors


eligible_symbols = [s for s in symbols if s not in symbol_failures]

if skip_phase2 or shared_from is None or not eligible_symbols:
    incremental_payload = {symbol: [] for symbol in symbols}
    print("Incremental fetch skipped.")
else:
    chunks = chunk_symbols(eligible_symbols, BATCH_SIZE)
    total_chunks = max(1, math.ceil(len(eligible_symbols) / BATCH_SIZE))
    incremental_payload = {symbol: [] for symbol in symbols}

    for idx, chunk in enumerate(chunks, start=1):
        pending_symbols = list(chunk)
        residual_errors: Dict[str, str] = {}

        for attempt in range(1, MAX_RETRIES + 1):
            chunk_data, chunk_errors = fetch_incremental_batch(pending_symbols, shared_from, shared_to)

            for symbol, rows in chunk_data.items():
                incremental_payload[symbol].extend(rows)

            pending_symbols = [s for s in pending_symbols if s in chunk_errors]
            residual_errors = {s: chunk_errors[s] for s in pending_symbols}

            if not pending_symbols:
                break

            if attempt < MAX_RETRIES:
                time.sleep(2 ** (attempt - 1))

        if residual_errors:
            failed_chunks.append({
                "chunk_index": idx,
                "symbols": list(residual_errors.keys()),
                "reason": "; ".join(f"{s}: {r}" for s, r in residual_errors.items()),
            })
            for symbol, reason in residual_errors.items():
                symbol_failures[symbol] = reason

        print(
            f"Chunk {idx}/{total_chunks} processed | symbols={len(chunk)} | failures={len(residual_errors)}"
        )

Chunk 1/1 processed | symbols=3 | failures=0


## Block 9 — Normalize, Merge, Deduplicate, Persist

### Purpose
Apply schema and quality rules, merge new rows into archives, and persist deterministic CSV outputs.

### Normalization rules
- Canonical columns: `Date, Open, High, Low, Close, Adjusted_close, Volume`.
- Parse/coerce date and numeric fields.
- Drop malformed rows (`Date` or `Close` missing after coercion).
- Keep fixed column order.

### Merge rules (per symbol)
1. Load existing archive if present.
2. Append normalized incoming rows.
3. Deduplicate on `Date` with `keep='last'` (new data wins).
4. Sort ascending by `Date`.
5. Compute `rows_before`, `rows_after`, `rows_added`.

### Persistence and status
- Write archive unless `DRY_RUN=True`.
- Populate `merge_stats`.
- Assign `status_by_symbol` as `bootstrapped`, `updated`, `skipped`, or `failed`.

### Output
Per-symbol merge/persist logs and updated run metrics.

In [9]:
CANONICAL_COLUMNS = ["Date", "Open", "High", "Low", "Close", "Adjusted_close", "Volume"]


def normalize_payload_to_rows(raw_payload: Any) -> pd.DataFrame:
    if isinstance(raw_payload, pd.DataFrame):
        df = raw_payload.copy()
    elif isinstance(raw_payload, list):
        df = pd.DataFrame(raw_payload)
    elif raw_payload is None:
        df = pd.DataFrame()
    else:
        raise TypeError("raw_payload must be DataFrame, list of dicts, or None.")

    rename_map = {
        "date": "Date",
        "Date": "Date",
        "open": "Open",
        "Open": "Open",
        "high": "High",
        "High": "High",
        "low": "Low",
        "Low": "Low",
        "close": "Close",
        "Close": "Close",
        "adjusted_close": "Adjusted_close",
        "adjustedClose": "Adjusted_close",
        "adjusted close": "Adjusted_close",
        "Adjusted close": "Adjusted_close",
        "Adjusted_close": "Adjusted_close",
        "volume": "Volume",
        "Volume": "Volume",
    }
    df = df.rename(columns={col: rename_map.get(col, col) for col in df.columns})

    for col in CANONICAL_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    for col in ["Open", "High", "Low", "Close", "Adjusted_close", "Volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["Date", "Close"]).copy()
    if df.empty:
        return pd.DataFrame(columns=CANONICAL_COLUMNS)

    df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
    df = df[CANONICAL_COLUMNS].copy()
    return df


def merge_and_dedupe_symbol_archive(symbol: str, incoming_rows: List[Dict[str, Any]], archive_path: Path) -> Dict[str, int]:
    incoming_df = normalize_payload_to_rows(incoming_rows)

    if archive_path.exists():
        existing_df = normalize_payload_to_rows(pd.read_csv(archive_path))
    else:
        existing_df = pd.DataFrame(columns=CANONICAL_COLUMNS)

    rows_before = int(len(existing_df))

    merged_df = pd.concat([existing_df, incoming_df], ignore_index=True)
    merged_df = normalize_payload_to_rows(merged_df)
    merged_df = merged_df.drop_duplicates(subset=["Date"], keep="last")
    merged_df = merged_df.sort_values(by="Date").reset_index(drop=True)

    rows_after = int(len(merged_df))
    rows_added = int(rows_after - rows_before)

    if not DRY_RUN:
        merged_df.to_csv(archive_path, index=False)

    return {"rows_before": rows_before, "rows_after": rows_after, "rows_added": rows_added}


for symbol in symbols:
    if symbol in symbol_failures and symbol not in incremental_payload:
        status_by_symbol[symbol] = "failed"
        continue

    incoming_rows = incremental_payload.get(symbol, [])

    try:
        if incoming_rows:
            stats = merge_and_dedupe_symbol_archive(symbol, incoming_rows, archive_map[symbol])
            merge_stats[symbol] = stats
            if symbol in bootstrap_success_symbols and stats["rows_before"] == 0:
                status_by_symbol[symbol] = "bootstrapped"
            elif stats["rows_added"] > 0:
                status_by_symbol[symbol] = "updated"
            else:
                status_by_symbol[symbol] = "skipped"
        else:
            current_rows = 0
            if archive_map[symbol].exists():
                current_rows = int(len(pd.read_csv(archive_map[symbol])))
            merge_stats.setdefault(symbol, {"rows_before": current_rows, "rows_after": current_rows, "rows_added": 0})
            if symbol in symbol_failures:
                status_by_symbol[symbol] = "failed"
            elif symbol in bootstrap_success_symbols:
                status_by_symbol[symbol] = "bootstrapped"
            else:
                status_by_symbol[symbol] = "skipped"

        stats = merge_stats[symbol]
        print(
            f"{symbol}: rows {stats['rows_before']} -> {stats['rows_after']} (delta={stats['rows_added']}) | status={status_by_symbol[symbol]}"
        )
    except Exception as exc:
        symbol_failures[symbol] = str(exc)
        status_by_symbol[symbol] = "failed"
        existing_rows = int(len(pd.read_csv(archive_map[symbol]))) if archive_map[symbol].exists() else 0
        merge_stats.setdefault(symbol, {"rows_before": existing_rows, "rows_after": existing_rows, "rows_added": 0})
        print(f"{symbol}: merge failed | {exc}")

AAPL.US: rows 11423 -> 11428 (delta=5) | status=updated
TSLA.US: rows 3970 -> 3975 (delta=5) | status=updated
AMZN.US: rows 7271 -> 7276 (delta=5) | status=updated


## Block 10 — Final Run Report

### Purpose
Aggregate run telemetry into a symbol-level report and a concise operational summary.

### Report columns
- `Symbol`
- `Status` (`bootstrapped`, `updated`, `skipped`, `failed`)
- `rows_before`
- `rows_after`
- `rows_added`
- `failure_reason`

### Summary output
Print totals for symbols by status, total rows added, and failed chunk count.

### Rerun support
If any symbol failed, export `failed_symbols.csv` to `SCOPE_FOLDER` for targeted reruns.

In [10]:
run_rows: List[Dict[str, Any]] = []
for symbol in symbols:
    stats = merge_stats.get(symbol, {"rows_before": 0, "rows_after": 0, "rows_added": 0})
    status = status_by_symbol.get(symbol)
    if status is None:
        status = "failed" if symbol in symbol_failures else "skipped"

    run_rows.append(
        {
            "Symbol": symbol,
            "Status": status,
            "rows_before": int(stats["rows_before"]),
            "rows_after": int(stats["rows_after"]),
            "rows_added": int(stats["rows_added"]),
            "failure_reason": symbol_failures.get(symbol, ""),
        }
    )

run_report_df = pd.DataFrame(run_rows)

status_counts = run_report_df["Status"].value_counts().to_dict() if not run_report_df.empty else {}
total_rows_added = int(run_report_df["rows_added"].sum()) if not run_report_df.empty else 0

print("Run summary:")
print(f"  total symbols: {len(run_report_df)}")
print(f"  bootstrapped: {status_counts.get('bootstrapped', 0)}")
print(f"  updated: {status_counts.get('updated', 0)}")
print(f"  skipped: {status_counts.get('skipped', 0)}")
print(f"  failed: {status_counts.get('failed', 0)}")
print(f"  total rows added: {total_rows_added}")
print(f"  failed chunks: {len(failed_chunks)}")

if run_report_df.empty:
    print("No rows to display in run report.")
else:
    print(run_report_df.to_string(index=False))

failed_df = run_report_df[run_report_df["Status"] == "failed"][["Symbol"]].copy()
if not failed_df.empty:
    failed_symbols_path = SCOPE_FOLDER / "failed_symbols.csv"
    if not DRY_RUN:
        failed_df.to_csv(failed_symbols_path, index=False)
        print(f"Failed symbol list exported to: {failed_symbols_path}")
    else:
        print(f"DRY_RUN=True, failed symbol list not written: {failed_symbols_path}")
else:
    print("No failed symbols, no rerun file generated.")

Run summary:
  total symbols: 3
  bootstrapped: 0
  updated: 3
  skipped: 0
  failed: 0
  total rows added: 15
  failed chunks: 0
 Symbol  Status  rows_before  rows_after  rows_added failure_reason
AAPL.US updated        11423       11428           5               
TSLA.US updated         3970        3975           5               
AMZN.US updated         7271        7276           5               
No failed symbols, no rerun file generated.
